In [ ]:
%%capture
!pip install keras-cv

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow" # other options: tensorflow or torch
import json
import keras_cv
import keras
from keras import ops
import tensorflow as tf

import cv2
import pandas as pd
import numpy as np
from glob import glob
from tqdm.notebook import tqdm
import joblib

import matplotlib.pyplot as plt

In [ ]:
class CFG:
    verbose = 1  # Verbosity
    seed = 47  # Random seed
    neg_sample = 0.01 # Downsample negative calss
    pos_sample = 5.0  # Upsample positive class
    preset = "efficientnetv2_b2_imagenet"  # Name of pretrained classifier
    image_size = [128, 128]  # Input image size
    epochs = 10 # Training epochs
    batch_size = 256  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    class_names = ['target']
    num_classes = 1

keras.utils.set_random_seed(CFG.seed)

In [ ]:
ROOT = str(os.getcwd())
BASE_PATH = f"{ROOT}/drive/MyDrive/WAIT"
DATA_PATH = f"{BASE_PATH}/data/new"
MODEL_PATH = f"{BASE_PATH}/models/enhanced/"
PROC_PATH = f"{MODEL_PATH}/enhanced/preprocessed_dataset/"

In [ ]:
# Train + Valid
df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_train_metadata_clean.parquet').ffill()

# Testing
testing_df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_test_metadata_clean.parquet').ffill()

In [ ]:
print("Class Distribution Before Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

# Sampling
positive_df = df.query("target==0").sample(frac=CFG.neg_sample, random_state=CFG.seed)
negative_df = df.query("target==1").sample(frac=CFG.pos_sample, replace=True, random_state=CFG.seed)
df = pd.concat([positive_df, negative_df], axis=0).sample(frac=1.0)

print("\nCalss Distribution After Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

Class Distribution Before Sampling (%):


,proportion
target,
0,99.899858
1,0.100142



Calss Distribution After Sampling (%):


,proportion
target,
0,66.611631
1,33.388369


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(df['target']), y=df['target'])
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

Class Weights: {0: np.float64(0.7506196640044065), 1: np.float64(1.4975274725274725)}


In [ ]:
import h5py

training_validation_hdf5 = h5py.File(f"{DATA_PATH}/train-image.hdf5", 'r')
testing_hdf5 = h5py.File(f"{DATA_PATH}/test-image.hdf5", 'r')

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True) # ensure continuous index
df["fold"] = -1
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
for i, (training_idx, validation_idx) in enumerate(sgkf.split(df, y=df.target, groups=df.patient_id)):
    df.loc[validation_idx, "fold"] = int(i)

# Use first fold for training and validation
training_df = df.query("fold!=0")
validation_df = df.query("fold==0")
print(f"# Num Train: {len(training_df)} | Num Valid: {len(validation_df)}")

# Num Train: 4383 | Num Valid: 1068


In [ ]:
# class distribution in training and validation
print("training:", training_df.target.value_counts(),
      "\nvalidation:", validation_df.target.value_counts(), sep='\n')

training:
target
0    2929
1    1454
Name: count, dtype: int64

validation:
target
0    702
1    366
Name: count, dtype: int64


# 📊 | Tabular Features

### Enhanced


In [ ]:
# Categorical features which will be one hot encoded
CATEGORICAL_COLUMNS = ["sex", "anatom_site_general", "tbp_tile_type",
                       "tbp_lv_location", "attribution"]

NEW_CATEGORICAL_COLUMNS = ["combined_anatomical_site"]

# Numerical features which will be normalized
NUMERIC_COLUMNS = ["age_approx", "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
           "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio", "tbp_lv_color_std_mean",
           "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM"]

NEW_NUMERIC_COLUMNS = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
         "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency", "color_consistency",

        "size_age_interaction", "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation", "overall_color_difference",
        "symmetry_perimeter_interaction", "comprehensive_lesion_index"]

# Tabular feature columns
FEAT_COLS = (CATEGORICAL_COLUMNS + NEW_CATEGORICAL_COLUMNS) + (NUMERIC_COLUMNS + NEW_NUMERIC_COLUMNS)

In [ ]:
def build_augmenter():
    # Define augmentations
    aug_layers = [
        keras_cv.layers.RandomCutout(height_factor=(0.02, 0.06), width_factor=(0.02, 0.06)),
        keras_cv.layers.RandomFlip(mode="horizontal"),
    ]

    # Apply augmentations to random samples
    aug_layers = [keras_cv.layers.RandomApply(x, rate=0.5) for x in aug_layers]

    # Build augmentation layer
    augmenter = keras_cv.layers.Augmenter(aug_layers)

    # Apply augmentations
    def augment(inp, label):
        images = inp["images"]
        aug_data = {"images": images}
        aug_data = augmenter(aug_data)
        inp["images"] = aug_data["images"]
        return inp, label
    return augment


def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        # Read jpeg image
        file_bytes = inp["images"]
        image = tf.io.decode_jpeg(file_bytes)

        # Resize
        image = tf.image.resize(image, size=target_size, method="area")

        # Rescale image
        image = tf.cast(image, tf.float32)
        image /= 255.0

        # Reshape
        image = tf.reshape(image, [*target_size, 3])

        inp["images"] = image
        return inp

    def decode_label(label, num_classes):
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [num_classes])
        return label

    def decode_with_labels(inp, label=None):
        inp = decode_image(inp)
        label = decode_label(label, CFG.num_classes)
        return (inp, label)

    return decode_with_labels if with_labels else decode_image


def build_dataset(
    isic_ids,
    hdf5,
    features,
    labels=None,
    batch_size=32,
    decode_fn=None,
    augment_fn=None,
    augment=False,
    shuffle=1024,
    cache=True,
    drop_remainder=False,
):
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)

    if augment_fn is None:
        augment_fn = build_augmenter()

    AUTO = tf.data.experimental.AUTOTUNE

    images = [None]*len(isic_ids)
    for i, isic_id in enumerate(tqdm(isic_ids, desc="Loading Images ")):
        images[i] = hdf5[isic_id][()]

    inp = {"images": images, "features": features}
    slices = (inp, labels) if labels is not None else inp

    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.cache() if cache else ds
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)
    if shuffle:
        ds = ds.shuffle(shuffle, seed=CFG.seed)
        opt = tf.data.Options()
        opt.deterministic = False
        ds = ds.with_options(opt)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder)
    ds = ds.map(augment_fn, num_parallel_calls=AUTO) if augment else ds
    ds = ds.prefetch(AUTO)
    return ds

In [ ]:
# Build Training & Validation Dataset
print("# Training:")
training_features = dict(training_df[FEAT_COLS])
training_ids = training_df.isic_id.values
training_labels = training_df.target.values
training_ds = build_dataset(training_ids, training_validation_hdf5, training_features,
                         training_labels, batch_size=CFG.batch_size,
                         shuffle=True, augment=False)

print("# Validation:")
validation_features = dict(validation_df[FEAT_COLS])
validation_ids = validation_df.isic_id.values
validation_labels = validation_df.target.values
validation_ds = build_dataset(validation_ids, training_validation_hdf5, validation_features,
                         validation_labels, batch_size=CFG.batch_size,
                         shuffle=False, augment=False)

# Testing
print("# Testing:")
testing_features = dict(testing_df[FEAT_COLS])
testing_ids = testing_df.isic_id.values
testing_ds = build_dataset(testing_ids, testing_hdf5,
                        testing_features, batch_size=CFG.batch_size,
                         shuffle=False, augment=False, cache=False)

# Training:


Loading Images :   0%|          | 0/4383 [00:00<?, ?it/s]

# Validation:


Loading Images :   0%|          | 0/1068 [00:00<?, ?it/s]

# Testing:


Loading Images :   0%|          | 0/37576 [00:00<?, ?it/s]

In [ ]:
feature_space = keras.utils.FeatureSpace(
    features={
        # Categorical features encoded as integers
        "sex": "string_categorical",
        "anatom_site_general": "string_categorical",
        "tbp_tile_type": "string_categorical",
        "tbp_lv_location": "string_categorical",
        "combined_anatomical_site": "string_categorical",
        "attribution": "string_categorical",
        # Numerical features to discretize
        "age_approx": "float_discretized",
        # Numerical features to normalize
        "tbp_lv_nevi_confidence": "float_normalized",
        "clin_size_long_diam_mm": "float_normalized",
        "tbp_lv_areaMM2": "float_normalized",
        "tbp_lv_area_perim_ratio": "float_normalized",
        "tbp_lv_color_std_mean": "float_normalized",
        "tbp_lv_deltaLBnorm": "float_normalized",
        "tbp_lv_minorAxisMM": "float_normalized",
        "lesion_size_ratio" : "float_normalized",
        "lesion_shape_index" : "float_normalized",
        "hue_contrast" : "float_normalized",
        "luminance_contrast": "float_normalized",
        "lesion_color_difference": "float_normalized",
        "border_complexity": "float_normalized",
        "3d_position_distance": "float_normalized",
        "perimeter_to_area_ratio": "float_normalized",
        "lesion_visibility_score": "float_normalized",
        "symmetry_border_consistency": "float_normalized",
        "color_consistency": "float_normalized",
        "size_age_interaction": "float_normalized",
        "hue_color_std_interaction": "float_normalized",
        "lesion_severity_index": "float_normalized",
        "shape_complexity_index": "float_normalized",
        "color_contrast_index": "float_normalized",
        "log_lesion_area": "float_normalized",
        "mean_hue_difference": "float_normalized",
        "std_dev_contrast": "float_normalized",
        "color_shape_composite_index": "float_normalized",
        "3d_lesion_orientation": "float_normalized",
        "overall_color_difference": "float_normalized",
        "symmetry_perimeter_interaction": "float_normalized",
        "comprehensive_lesion_index": "float_normalized",
        # numerical features already normalized
        "normalized_lesion_size": "float",
        },
    output_mode="concat",
)

In [ ]:
training_ds_with_no_labels = training_ds.map(lambda x, _: x["features"])
feature_space.adapt(training_ds_with_no_labels)
print(feature_space.get_encoded_features().shape)

(None, 128)


In [ ]:
# fs_dir = os.path.join(MODEL_PATH, "model-metadata")
# os.makedirs(fs_dir, exist_ok=True)
# filepath = os.path.join(fs_dir, "feature_space_test.keras")

# feature_space.save(filepath)
# print(f"FeatureSpace saved at: '{filepath}'")

In [ ]:
# Apply Feature Processing

training_ds = training_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

validation_ds = validation_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

testing_ds = testing_ds.map(
    lambda x: {"images": x["images"],
               "features": feature_space(x["features"])}, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
# loss & metric
auc = keras.metrics.AUC()
loss = keras.losses.BinaryCrossentropy(label_smoothing=0.02)

In [ ]:
# Define input layers
image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
inp = {"images":image_input, "features":feat_input}

# Branch for image input
backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
x1 = backbone(image_input)
x1 = keras.layers.GlobalAveragePooling2D()(x1)
x1 = keras.layers.Dropout(0.2)(x1)

# Branch for tabular/feature input
x2 = keras.layers.Dense(96, activation="selu")(feat_input)
x2 = keras.layers.Dense(128, activation="selu")(x2)
x2 = keras.layers.Dropout(0.1)(x2)

# Concatenate both branches
concat = keras.layers.Concatenate()([x1, x2])

# Output layer
out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

# Build model
model = keras.models.Model(inp, out)

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=loss,
    metrics=[auc],
)

# Model Summary
model.summary()

100%|██████████| 1.79k/1.79k [00:00<00:00, 3.03MB/s]


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ images (InputLayer) │ (None, 128, 128,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ features            │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficient_net_v2b2… │ (None, 4, 4,      │  8,769,374 │ images[0][0]      │
│ (EfficientNetV2Bac… │ 1408)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 96)        │     12,384 │ features[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1408)      │          0 │ efficient_net_v2… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     12,416 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1408)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1536)      │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │      1,537 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,795,711 (33.55 MB)

 Trainable params: 8,713,423 (33.24 MB)

 Non-trainable params: 82,288 (321.44 KB)

In [ ]:
# keras.utils.plot_model(model, show_shapes=True, show_layer_names=True, dpi=60)

In [ ]:
import math

def get_lr_callback(batch_size=8, mode='cos', epochs=10, plot=False):
    lr_start, lr_max, lr_min = 2.5e-5, 5e-6 * batch_size, 0.8e-5
    lr_ramp_ep, lr_sus_ep, lr_decay = 3, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if epoch < lr_ramp_ep: lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep: lr = lr_max
        elif mode == 'exp': lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
        elif mode == 'step': lr = lr_max * lr_decay**((epoch - lr_ramp_ep - lr_sus_ep) // 2)
        elif mode == 'cos':
            decay_total_epochs, decay_epoch_index = epochs - lr_ramp_ep - lr_sus_ep + 3, epoch - lr_ramp_ep - lr_sus_ep
            phase = math.pi * decay_epoch_index / decay_total_epochs
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:  # Plot lr curve if plot is True
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(epoch) for epoch in np.arange(epochs)], marker='o')
        plt.xlabel('epoch'); plt.ylabel('lr')
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=False)  # Create lr callback

lr_cb = get_lr_callback(CFG.batch_size, mode="exp", plot=False)

In [ ]:
%%capture
!pip install --no-cache-dir tf-explain

In [ ]:
raise StopExecution

NameError: name 'asd' is not defined

In [ ]:
""" Module for image operations """
import numpy as np
import tensorflow as tf


def apply_grey_patch(image, top_left_x, top_left_y, patch_size):
    """
    Replace a part of the image with a grey patch.

    Args:
        image (numpy.ndarray): Input image
        top_left_x (int): Top Left X position of the applied box
        top_left_y (int): Top Left Y position of the applied box
        patch_size (int): Size of patch to apply

    Returns:
        numpy.ndarray: Patched image
    """
    patched_image = np.array(image, copy=True)
    patched_image[
        top_left_y : top_left_y + patch_size, top_left_x : top_left_x + patch_size, :
    ] = 127.5

    return patched_image


@tf.function
def transform_to_normalized_grayscale(tensor):
    """ Transform tensor over RGB axis to grayscale and apply min-max normalization per image. """
    grayscale_tensor = tf.reduce_sum(tensor, axis=-1)

    # min-max normalization per image
    v_min = tf.reduce_min(grayscale_tensor, axis=(1, 2), keepdims=True)
    v_max = tf.reduce_max(grayscale_tensor, axis=(1, 2), keepdims=True)
    normalized_tensor = (grayscale_tensor - v_min) / (v_max - v_min + 1e-8)

    return normalized_tensor


def grid_display(array, num_rows=None, num_columns=None):
    """
    Display a list of images as a grid.

    Args:
        array (numpy.ndarray): 4D Tensor (batch_size, height, width, channels)

    Returns:
        numpy.ndarray: 3D Tensor as concatenation of input images on a grid
    """
    if num_rows is not None and num_columns is not None:
        total_grid_size = num_rows * num_columns
        if total_grid_size < len(array):
            warnings.warn(
                Warning(
                    "Given values for num_rows and num_columns doesn't allow to display "
                    "all images. Values have been overrided to respect at least num_columns"
                )
            )
            num_rows = math.ceil(len(array) / num_columns)
    elif num_rows is not None:
        num_columns = math.ceil(len(array) / num_rows)
    elif num_columns is not None:
        num_rows = math.ceil(len(array) / num_columns)
    else:
        num_rows = math.ceil(math.sqrt(len(array)))
        num_columns = math.ceil(math.sqrt(len(array)))

    number_of_missing_elements = num_columns * num_rows - len(array)
    # We fill the array with np.zeros elements to obtain a perfect square
    array = np.append(
        array,
        np.zeros((number_of_missing_elements, *array[0].shape)).astype(array.dtype),
        axis=0,
    )

    grid = np.concatenate(
        [
            np.concatenate(
                array[index * num_columns : (index + 1) * num_columns], axis=1
            )
            for index in range(num_rows)
        ],
        axis=0,
    )

    return grid

In [ ]:
import os
from pathlib import Path
from typing import Optional, Tuple, List, Union

import cv2
import numpy as np
import tensorflow as tf
from keras import Model
from tf_explain.utils.display import grid_display


class MultiModalSmoothGradEngine:
    """
    SmoothGrad engine for multimodal models (images + tabular features).

    Features:
    - Two modes:
        * 'context'     → uses real tabular features.
        * 'image_only'  → replaces features with a baseline (zeros or provided vector).
    - Output:

    - Visualization:
        * Accepts a separate display_images batch or a deprocess_fn to convert model-input images to [0,1].
    """

    def __init__(
        self,
        use_guided_grads: bool = True,
        mode: str = 'context',              # 'context' or 'image_only'
        baseline: Optional[np.ndarray] = None,
        deprocess_fn: Optional[callable] = None,  # fn(img_float) -> float [0,1]
    ):
        assert mode in ('context', 'image_only'), "mode must be 'context' or 'image_only'"
        self.mode = mode
        self.baseline = baseline
        self.deprocess_fn = deprocess_fn

    def explain(
        self,
        validation_data: Tuple[dict, np.ndarray],  # ({'images', 'features'}, labels)
        model: tf.keras.Model,
        class_index: int,
        num_samples: int = 5,
        noise: float = 1.0,
    ):
        """
        Compute SmoothGrad for a batch and return (grid, triptychs, preds).
        validation_data is # ({'images', 'features'}, labels)
        """
        # --- Unpack ---
        x, labels = validation_data
        images = x['images']
        features = x['features']

        # Generate noisy images: shape (batch * num_samples, H, W, 3)
        noisy_images = self.generate_noisy_images(images, num_samples, noise)

        # Tile features to match: shape (batch * num_samples, feature_dim)
        repeated_features = tf.repeat(features, num_samples, axis=0)

        # --- Build feeds in model.inputs order ---
        feeds = self._ordered_feeds(model, images=noisy_images, features=repeated_features)
        if self.mode == 'image_only':
            feeds = self._apply_baseline_to_feeds(model, feeds, images=noisy_images, features=repeated_features)

        smoothed_gradients = self.get_averaged_gradients(feeds, model, class_index, num_samples)

        grayscale_gradients = transform_to_normalized_grayscale(tf.abs(smoothed_gradients)).numpy()

       # Calculate predictions for triptych text (using original images)
        orig_feeds = self._ordered_feeds(model, images=images, features=features)
        preds = model(orig_feeds, training=False)
        preds_np = preds.numpy()

        # Generate Triptychs
        triptychs = []
        for i in range(images.shape[0]):
            pred_text = self._format_pred_text(preds_np[i])
            # grayscale_gradients has shape (batch, H, W) after transform
            trip = self._compose_triptych(images[i], grayscale_gradients[i], pred_text)
            triptychs.append(trip)

        grid = self._make_grid(triptychs)
        return grid, triptychs, preds_np


    @staticmethod
    def generate_noisy_images(images, num_samples, noise):
        """
        Generate num_samples noisy images with std noise for each image.

        Args:
            images (numpy.ndarray): 4D-Tensor with shape (batch_size, H, W, 3)
            num_samples (int): Number of noisy samples to generate for each input image
            noise (float): Standard deviation for noise normal distribution

        Returns:
            np.ndarray: 4D-Tensor of noisy images with shape (batch_size*num_samples, H, W, 3)
        """
        repeated_images = np.repeat(images, num_samples, axis=0)
        noise = np.random.normal(0, noise, repeated_images.shape).astype(np.float32)

        return repeated_images + noise


    def get_averaged_gradients(self, feeds, model, class_index, num_samples):
        """
        Compute average of gradients for target class.
        """
        # Identify which input in feeds is the image tensor
        image_input_index = next(i for i, kt in enumerate(model.inputs) if len(kt.shape) == 4)

        # --- Forward + gradients ---
        with tf.GradientTape() as tape:
            # watch the specific image tensor inside feeds
            img_tensors = tf.cast(feeds[image_input_index], tf.float32)
            tape.watch(img_tensors)

            # Update feeds with the watched tensor
            current_feeds = list(feeds)
            current_feeds[image_input_index] = img_tensors

            preds = model(current_feeds, training=False)

            # target score, use logit(p) to avoid saturation artifacts
            if preds.shape[-1] == 1:
                p = tf.clip_by_value(preds[:, 0], 1e-6, 1.0 - 1e-6)
                target = tf.math.log(p) - tf.math.log(1.0 - p)  # (B,)
            else:
                # multi-class / multi-label
                p = tf.clip_by_value(preds[:, class_index], 1e-6, 1.0 - 1e-6)
                target = tf.math.log(p)

        grads = tape.gradient(target, img_tensors)  # (B, Hc, Wc, C)

        grads_per_image = tf.reshape(grads, (-1, num_samples, *grads.shape[1:]))
        averaged_grads = tf.reduce_mean(grads_per_image, axis=1)

        return averaged_grads

    # ------------------------ Saving helpers ------------------------


    def save_grid_grayscale(self, grid: np.ndarray, output_dir: str, output_name: str) -> None:
        """Save the grid image to disk (RGB)."""
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        out_path = str(Path(output_dir) / output_name)
        img_u8 = np.uint8(255 * np.clip(grid, 0, 1))
        cv2.imwrite(out_path, cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR))

    def save_triptychs(
        self,
        triptychs: List[np.ndarray],
        base_dir: str,
        epoch_name: str,
        prefix: str = "sample",
    ) -> None:
        """
        Save each triptych as:
          base_dir/epoch_name/{prefix}_{i:03d}.png
        """
        out_dir = Path(base_dir) / epoch_name
        out_dir.mkdir(parents=True, exist_ok=True)
        for i, img in enumerate(triptychs):
            img_u8 = np.uint8(255 * np.clip(img, 0, 1))
            path = out_dir / f"{prefix}_{i:03d}.png"
            cv2.imwrite(str(path), cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR))

    # ------------------------ Internal helpers ------------------------
    def _make_grid(self, triptychs: List[np.ndarray]) -> np.ndarray:
        """uses tf-explain grid_display to create a grid"""
        thumbs = np.stack([np.uint8(255 * np.clip(t, 0, 1)) for t in triptychs], axis=0)
        canvas = grid_display(thumbs)  # returns uint8 RGB
        return canvas.astype(np.float32) / 255.0


    def _resolve_target_tensor(self, model: tf.keras.Model, layer_name: Optional[str]):
        """
        Prefer the 4D tensor feeding the first GlobalAveragePooling2D.
        If layer_name is provided, use that layer's output; else fallback to last 4D layer.
        """
        if layer_name is not None:
            return model.get_layer(layer_name).output
        # Prefer the tensor feeding the first GAP
        for l in model.layers:
            if isinstance(l, tf.keras.layers.GlobalAveragePooling2D):
                return l.input
        # Fallback to last 4D tensor in the graph
        for l in reversed(model.layers):
            try:
                shp = l.output_shape
            except Exception:
                continue
            if isinstance(shp, tuple) and len(shp) == 4:
                return l.output
        raise ValueError("No suitable 4D conv tensor found for Grad-CAM target")

    def _ordered_feeds(self, model: tf.keras.Model, images, features):
        """Create a list of inputs respecting model.inputs order (rank-4 → images, rank-2 → features)."""
        feeds = []
        for kt in model.inputs:
            if len(kt.shape) == 4:
                feeds.append(images)
            elif len(kt.shape) == 2:
                feeds.append(features)
            else:
                raise ValueError(f"Unexpected input rank for {kt.name}: {kt.shape}")
        return feeds

    def _apply_baseline_to_feeds(self, model: tf.keras.Model, feeds, images, features):
        """Replace the features tensor in feeds with a baseline (zeros by default)."""
        feat_inp = next(inp for inp in model.inputs if len(inp.shape) == 2)
        F = int(feat_inp.shape[-1])
        if self.baseline is None:
            baseline = tf.zeros((tf.shape(images)[0], F), dtype=getattr(features, 'dtype', tf.float32))
        else:
            base = tf.convert_to_tensor(self.baseline)
            if base.shape.rank == 1:
                baseline = tf.broadcast_to(base[None, :], [tf.shape(images)[0], F])
            else:
                baseline = base
        feeds_out = []
        for kt, fed in zip(model.inputs, feeds):
            feeds_out.append(baseline if len(kt.shape) == 2 else fed)
        return feeds_out

    def _generate_cams(self, conv_outputs: tf.Tensor, grads: tf.Tensor) -> List[np.ndarray]:
        """Compute per-sample weighted sum of conv activations; returns list of 2D CAMs normalized to [0,1]."""
        cams = []
        conv_np = conv_outputs.numpy() if isinstance(conv_outputs, tf.Tensor) else conv_outputs
        grads_np = grads.numpy() if isinstance(grads, tf.Tensor) else grads
        B = conv_np.shape[0]
        for i in range(B):
            weights = grads_np[i].mean(axis=(0, 1), keepdims=False)  # (C,)
            # weights = grads_np[i][1:-1, 1:-1, :].mean(axis=(0, 1))
            cam = (conv_np[i] * weights[None, None, :]).sum(axis=-1)
            cam = np.maximum(cam, 0.0)
            cam_min, cam_max = cam.min(), cam.max()
            cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)
            cams.append(cam)
        return cams

    def _format_pred_text(self, pred_row):
        """Format prediction text for display: binary → p=..., multi-class → ŷ=k (p=...)."""
        try:
            arr = np.array(pred_row)
        except Exception:
            return f"p={float(pred_row):.3f}"
        if arr.ndim == 0:
            return f"p={float(arr):.3f}"
        arr = arr.ravel()
        if arr.size == 1:
            return f"p={float(arr[0]):.3f}"
        else:
            k = int(np.argmax(arr))
            p = float(arr[k])
            return f"ŷ={k} (p={p:.3f})"

    def _compose_triptych(self, image_like: np.ndarray, cam_2d: np.ndarray, pred_text: str, colormap=cv2.COLORMAP_JET, image_weight=0.8) -> np.ndarray:
        """Create a 3-panel image (orig+text | heatmap | overlay), each panel HxWx3 in [0,1]."""
        img = self._to_display_rgb(image_like)
        H, W = img.shape[:2]
        # Heatmap
        heat = cv2.applyColorMap(np.uint8(255 * cv2.resize(cam_2d, (W, H))),colormap)
        heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB) / 255.0
        # Overlay
        overlay = image_weight * img + (1.0 - image_weight) * heat
        overlay = np.clip(overlay, 0, 1)
        # Original with prediction text
        orig = (img.copy() * 255).astype(np.uint8)
        txt = pred_text if isinstance(pred_text, str) else str(pred_text)
        cv2.putText(orig, txt, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(orig, txt, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
        orig = orig.astype(np.float32) / 255.0
        return np.hstack([orig, heat, overlay])

    # ---------------- Display helpers ----------------
    def _to_display_rgb(self, image: np.ndarray) -> np.ndarray:
        """
        Convert a model-input image to float RGB in [0,1] for visualization.
        Applies custom deprocess_fn if provided; else uses heuristics.
        """
        arr = image.numpy() if tf.is_tensor(image) else np.asarray(image)

        if self.deprocess_fn is not None:
            out = self.deprocess_fn(arr)
            return np.clip(self._ensure_rgb(out), 0.0, 1.0)

        # 1) uint8 → [0,1]
        if arr.dtype == np.uint8:
            arr = arr.astype(np.float32) / 255.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # 2) Float ranges
        arr = arr.astype(np.float32)
        amin, amax = float(arr.min()), float(arr.max())

        # true [-1,1] only if there are negatives
        if amin < 0.0 and amax <= 1.1:
            arr = (arr + 1.0) / 2.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # already [0,1]
        if 0.0 <= amin and amax <= 1.1:
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # [0,255] floats
        if amin >= -1.0 and amax > 1.5 and amax <= 260:
            arr = arr / 255.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # fallback: per-image min-max
        rng = (amax - amin) if (amax - amin) > 1e-8 else 1.0
        arr = (arr - amin) / rng
        return np.clip(self._ensure_rgb(arr), 0, 1)

    def _ensure_rgb(self, arr: np.ndarray) -> np.ndarray:
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)
        return arr

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf
import cv2
from keras.callbacks import Callback

class MultiModalSmoothGradCallback(Callback):
    def __init__(self, engine, probe_batch, class_index=0,
                 tb_dir="./logs/smoothgrad", save_dir=None, tag="SmoothGrad/grid"):
        super().__init__()
        self.engine = engine
        self.probe_batch = probe_batch
        self.class_index = class_index
        self.tag = tag
        self.writer = tf.summary.create_file_writer(tb_dir)
        self.save_dir = Path(save_dir) if save_dir else None
        if self.save_dir:
            self.save_dir.mkdir(parents=True, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        grid, _, _ = self.engine.explain(
            validation_data=self.probe_batch,
            model=self.model,
            class_index=self.class_index,
        )

        # TensorBoard
        img = tf.clip_by_value(tf.convert_to_tensor(grid, tf.float32), 0.0, 1.0)[None, ...]
        with self.writer.as_default():
            tf.summary.image(self.tag, img, step=epoch)
            self.writer.flush()

        # Disk (PNG)
        if self.save_dir:
            out = (np.clip(grid, 0, 1) * 255).astype(np.uint8)
            cv2.imwrite(str(self.save_dir / f"epoch_{epoch:03d}.png"),
                        cv2.cvtColor(out, cv2.COLOR_RGB2BGR))


In [ ]:
XAI_CB_PATH = os.path.join(MODEL_PATH, 'xai', 'test', 'no_aug' , 'callback')
base_dir=os.path.join(XAI_CB_PATH, "images")
log_dir=os.path.join(XAI_CB_PATH, "tb")
os.makedirs(base_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

x, labels = next(iter(validation_ds.skip(0).take(1)))

gm_cb = MultiModalGradCAMCallback(
    engine=MultiModalGradCAMEngine(use_guided_grads=True, mode="image_only"),
    probe_batch=(x, labels),
    class_index=0,
    tb_dir=log_dir,
    save_dir=base_dir,
    tag="GradCAM/grid",
)

In [ ]:
model_fpath = os.path.join(MODEL_PATH, 'saved_models', 'best_enhanced_model_test_noaug.keras')

ckpt_cb = keras.callbacks.ModelCheckpoint(
    model_fpath,
    monitor="val_auc",
    save_best_only=True,
    save_weights_only=False,
    mode="max",
)

In [ ]:
cbs = [lr_cb, ckpt_cb, gm_cb]

history = model.fit(
    training_ds,
    epochs=4,
    callbacks=cbs,
    validation_data=validation_ds,
    verbose=CFG.verbose,
    class_weight=class_weights,
)

Epoch 1/4
18/18 ━━━━━━━━━━━━━━━━━━━━ 271s 7s/step - auc: 0.4968 - loss: 0.7045 - val_auc: 0.5309 - val_loss: 0.7062 - learning_rate: 2.5000e-05
Epoch 2/4
18/18 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - auc: 0.8203 - loss: 0.5543 - val_auc: 0.8473 - val_loss: 0.5109 - learning_rate: 4.4333e-04
Epoch 3/4
18/18 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - auc: 0.9683 - loss: 0.2729 - val_auc: 0.8284 - val_loss: 0.5124 - learning_rate: 8.6167e-04
Epoch 4/4
18/18 ━━━━━━━━━━━━━━━━━━━━ 25s 1s/step - auc: 0.9953 - loss: 0.1317 - val_auc: 0.8699 - val_loss: 0.4767 - learning_rate: 0.0013


In [ ]:
ds

In [ ]:
# %reload_ext tensorboard
# %tensorboard --logdir "{os.path.join(XAI_PATH, 'tb')}"

In [ ]:
# model.save(model_fpath, include_optimizer=True)

test flip

In [ ]:
# 1) Instantiate the engine
engine = MultiModalGradCAMEngine(
    use_guided_grads=True,        # guided-ReLU style gradients
    mode='context',               # 'context' or 'image_only'
    baseline=None,                # for image_only; defaults to zeros per feature width
)

debug

In [ ]:
test_path = '/content/drive/MyDrive/WAIT/models/enhanced/xai/grad_cam/nuovi_test/debug/callback'

(x0, y0) = next(iter(validation_ds.take(1)))

grid_o, triptychs_o, _ = engine.explain((x0, y0), model=model, class_index=0)
Ht, Wt, Ct = triptychs_o[0].shape

x1 = {"images": tf.image.flip_left_right(x0["images"]),
    "features": x0["features"]}

grid_f, _, _ = engine.explain((x1, y0), model=model, class_index=0)

engine.save_grid(grid_o, test_path, "orig.png")
engine.save_grid(grid_f, test_path, "flipped.png")
print("Triptych (tile) size:", (Ht, Wt, Ct))

In [ ]:
test_path = '/content/drive/MyDrive/WAIT/models/enhanced/xai/grad_cam/nuovi_test/debug/callback'

n_samp = 15
if n_samp is None:
    n_samp = len(validation_ds)

(x0, y0) = next(iter(validation_ds.take(1)))
x0 = {k: v[:n_samp] for k, v in x0.items()}
y0 = y0[:n_samp]

grid_o, triptychs_o, _ = engine.explain((x0, y0), model=model, class_index=0)

x1 = {"images": tf.image.flip_left_right(x0["images"]), "features": x0["features"]}
grid_f, _, _ = engine.explain((x1, y0), model=model, class_index=0)

engine.save_grid(grid_o, test_path, "orig.png")

print("Triptych (tile) size:", triptychs_o[0].shape)